## Импорт библиотек и загрузка данных

In [1]:
import os
print(os.getcwd())
print(os.listdir())

d:\Programs\analysis.ipynb
['.git', 'ad_spend.csv', 'edtech_data.db', 'final_work_database.db', 'funnel_events.csv', 'main.ipynb', 'payments.csv', 'README.md', 'test_results.csv', 'trial_lessons.csv', 'users.csv']


In [2]:
import pandas as pd
import sqlite3
import os



# Создаем подключение к базе
conn = sqlite3.connect('edtech_data.db')

# Загружаем файлы напрямую (так как названия теперь нормальные)
# Заменяем read_sql на read_csv во второй строке загрузки
pd.read_csv('users.csv').to_sql('users', conn, if_exists='replace', index=False)
pd.read_csv('payments.csv').to_sql('payments', conn, if_exists='replace', index=False)
pd.read_csv('trial_lessons.csv').to_sql('trial_lessons', conn, if_exists='replace', index=False)
pd.read_csv('ad_spend.csv').to_sql('ad_spend', conn, if_exists='replace', index=False)
pd.read_csv('funnel_events.csv').to_sql('funnel_events', conn, if_exists='replace', index=False)
pd.read_csv('test_results.csv').to_sql('test_results', conn, if_exists='replace', index=False)

print("Все таблицы загружены!")

Все таблицы загружены!


# Все таблицы

In [3]:
query = "SELECT * FROM users LIMIT 1"
pd.read_sql(query, conn)

,user_id,first_visit_at,acquisition_channel,campaign,device,age_group,country,landing_page,user_goal
0,U00001,2026-02-05 14:14:00,YouTube,yt_pre_roll,mobile,18-24,Austria,trial_lesson,exam


In [4]:
query = "SELECT * FROM funnel_events LIMIT 1"
pd.read_sql(query, conn)

,event_id,user_id,event_time,event_name,step_order,acquisition_channel,campaign
0,E000001,U00001,2026-02-05 14:14:00,site_visit,1,YouTube,yt_pre_roll


In [5]:
query = "SELECT * FROM ad_spend LIMIT 1"
pd.read_sql(query, conn)

,date,acquisition_channel,campaign,spend_eur,impressions,clicks
0,2026-01-01,Instagram,ig_reels_beginner,31.45,5425,43


In [6]:
query = "SELECT * FROM trial_lessons LIMIT 1"
pd.read_sql(query, conn)

,user_id,trial_booked_at,trial_attended_at,trial_attended,teacher_rating
0,U00001,2026-02-06 04:48:00,2026-02-09 16:48:00,1,3.6


In [7]:
query = "SELECT * FROM payments LIMIT 50"
pd.read_sql(query, conn)

,payment_id,user_id,payment_date,plan_name,subscription_type,amount_eur,payment_status,refund_flag,acquisition_channel,campaign
0,P00001,U00010,2026-04-02,Basic Materials,monthly,29.0,success,0,YouTube,yt_long_form
1,P00002,U00028,2026-02-13,Basic Materials,monthly,29.0,success,0,Partner Referral,partner_creator
2,P00003,U00047,2026-03-27,Basic Materials,monthly,29.0,success,0,YouTube,yt_pre_roll
3,P00004,U00054,2026-02-02,Basic Materials,monthly,29.0,success,0,Telegram,tg_channel_post
4,P00005,U00058,2026-03-02,Basic Materials,monthly,29.0,success,0,YouTube,yt_teacher_expert
5,P00006,U00067,2026-03-20,Basic Materials,monthly,29.0,success,0,Google Ads,ga_competitor
6,P00007,U00074,2026-03-07,Intensive Plan,monthly,149.0,success,0,Partner Referral,partner_blog
7,P00008,U00075,2026-02-12,Basic Materials,monthly,29.0,failed,0,YouTube,yt_pre_roll
8,P00009,U00081,2026-01-02,Basic Materials,monthly,29.0,success,0,Instagram,ig_story_discount
9,P00010,U00082,2026-02-15,Speaking Club,monthly,49.0,success,0,YouTube,yt_long_form


In [8]:
query = "SELECT * FROM test_results LIMIT 1"
pd.read_sql(query, conn)

,user_id,test_started_at,test_completed_at,test_score,recommended_level,recommended_plan
0,U00001,2026-02-05 14:48:00,2026-02-05 15:13:00,66.0,B1,Speaking Club


# Подготовка данных 

## Проверка количества строк в каждой таблице

Вывод: Объем выборки статистически значим; исключен риск анализа по единичным случаям

In [9]:
query = """SELECT COUNT(*) AS strings_users
,(SELECT COUNT(*) FROM funnel_events ) AS strings_funnel_events
,(SELECT COUNT(*) FROM test_results) AS strings_test_results
,(SELECT COUNT(*) FROM trial_lessons) AS strings_trial_lessons
,(SELECT COUNT(*) FROM payments) AS strings_payments
,(SELECT COUNT(*) FROM ad_spend ) AS strings_ad_spend
 FROM users """
pd.read_sql(query, conn)

,strings_users,strings_funnel_events,strings_test_results,strings_trial_lessons,strings_payments,strings_ad_spend
0,1500,5570,986,361,163,1620


## Уникальность user_id в таблице users

Вывод: Все пользователи уникальны(Дубликаты юзеров создают искажение относительно conversion rate )

In [10]:
query = """
SELECT DISTINCT COUNT(*) AS unique_users
,(SELECT COUNT(*)  FROM users) AS not_unique_users 
FROM users 

"""
pd.read_sql(query, conn)

,unique_users,not_unique_users
0,1500,1500


## Есть ли пользователи в событиях, которых нет в users

Вывод: Аномальных ID не обнаружено(Перед проблемных мест в пути пользователя по группам,нужно убедиться, что мы работаем с одинаковыми пользователями и данные собраны и интерпретированы правильно)

In [11]:
query = "SELECT DISTINCT user_id FROM  funnel_events WHERE user_id not in(SELECT DISTINCT user_id FROM users) "
pd.read_sql(query, conn)

,user_id


## Пропуски в ключевых полях

Вывод:Аномальных пропусков не обнаружено,кроме случая если клиент не допроходил тест или не заканчивал пробный урок(Перед работой с данными нужно убедиться что данные полные,чтобы не было пустых значеный или непральных в выводах )

### Пропуски user_id

In [12]:
query = """SELECT COUNT(*) AS id_null_users
 ,(SELECT COUNT(*) FROM funnel_events WHERE user_id IS NULL ) AS id_null_funnel_events
 ,(SELECT COUNT(*) FROM payments WHERE user_id IS NULL) AS id_null_payments
 ,(SELECT COUNT(*) FROM trial_lessons WHERE user_id IS NULL) AS id_null_trial_lessons
 ,(SELECT COUNT(*) FROM test_results WHERE user_id IS NULL) AS id_null_test_results
 FROM users WHERE user_id IS NULL
 """
pd.read_sql(query, conn)

,id_null_users,id_null_funnel_events,id_null_payments,id_null_trial_lessons,id_null_test_results
0,0,0,0,0,0


### Пропуски event_name и event_time

In [13]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_name IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


In [14]:
query = "SELECT COUNT(*) FROM funnel_events WHERE event_time IS NULL"
pd.read_sql(query, conn) 

,COUNT(*)
0,0


### Пропуски acquisition_channel

In [15]:
query = """
SELECT COUNT(*) AS channel_null_funnel_events
,(SELECT COUNT(*) FROM payments WHERE acquisition_channel IS NULL) AS channel_null_payments
,(SELECT COUNT(*) FROM ad_spend WHERE acquisition_channel IS NULL) AS channel_null_ad_spend
,(SELECT COUNT(*) FROM users WHERE acquisition_channel IS NULL) AS channel_null_users
FROM funnel_events WHERE acquisition_channel IS NULL
"""
pd.read_sql(query, conn) 

,channel_null_funnel_events,channel_null_payments,channel_null_ad_spend,channel_null_users
0,0,0,0,0


### Пропуски payment_status and amount_eur 

In [16]:
query = "SELECT COUNT(*) FROM payments WHERE payment_status IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


In [17]:
query = "SELECT COUNT(*) FROM payments WHERE amount_eur  IS NULL"
pd.read_sql(query, conn)

,COUNT(*)
0,0


### Провека всех строк

In [18]:
trial_lessons = pd.read_sql("SELECT * FROM trial_lessons", conn)
print(trial_lessons.isnull().sum())
print()
test_results = pd.read_sql("SELECT * FROM test_results", conn)
print(test_results.isnull().sum())
print()
users = pd.read_sql("SELECT * FROM users", conn)
print(users.isnull().sum())
print()
ad_spend = pd.read_sql("SELECT * FROM ad_spend", conn)
print(ad_spend.isnull().sum())
print()
funnel_events = pd.read_sql("SELECT * FROM funnel_events", conn)
print(funnel_events.isnull().sum())
print()
payments = pd.read_sql("SELECT * FROM payments", conn)
print(payments.isnull().sum())

user_id               0
trial_booked_at       0
trial_attended_at    79
trial_attended        0
teacher_rating       79
dtype: int64

user_id                0
test_started_at        0
test_completed_at    136
test_score           136
recommended_level    136
recommended_plan     136
dtype: int64

user_id                0
first_visit_at         0
acquisition_channel    0
campaign               0
device                 0
age_group              0
country                0
landing_page           0
user_goal              0
dtype: int64

date                   0
acquisition_channel    0
campaign               0
spend_eur              0
impressions            0
clicks                 0
dtype: int64

event_id               0
user_id                0
event_time             0
event_name             0
step_order             0
acquisition_channel    0
campaign               0
dtype: int64

payment_id             0
user_id                0
payment_date           0
plan_name              0
subscripti

## Дубликаты событий

Вывод:Дубликатов не выяылено (Для работы с категориями событий нужно убедиться,что нет дубликатов).

In [19]:
query = """SELECT DISTINCT COUNT(*) AS unique_events
,(SELECT COUNT(*) FROM funnel_events ) AS not_unique_events
FROM funnel_events

 """
pd.read_sql(query, conn)

,unique_events,not_unique_events
0,5570,5570


## Корректность дат

Вывод: Аномалий с датами не обнаружено (Для коректного рассчёта по периодам)

In [20]:
query = """ 
WITH tb AS( SELECT user_id
,event_time AS t1
,LAG(event_time,1) OVER(PARTITION BY user_id ORDER BY event_time) AS t2
FROM funnel_events  )


SELECT user_id, t1 AS event_time FROM tb
 WHERE
        t1 < t2         

"""

pd.read_sql(query, conn)

,user_id,event_time


## Корректность Платежей

ВЫВОДЫ: 

1.Аномальных айди платежей не выявлено, система платежей не плодит дубликаты платежей

2.Пользователи у которых платёж не прошёл - не возвращались к оплате снова

3.Нет ни одного возврата

4.22.7% ошибок платежа

ГИПОТЕЗЫ:

1.Не работают возвраты

2.Для повторной оплаты система создаёт новый айди пользователя и клиент вводит некоректные данные

3.Недостаточное количество вариантов платежей для клиента, вследствии чего нет попыток повторной оплаты

4.Процент неудачных платежей весомый,соотвественно ошибки в платежах вызваны не только пользователями,но и некоректной работы терминала оплаты

РЕКОМЕНДАЦИИ:

1.Проверить коректность работы возвратов

2.Проверить коректность работы терминала оплаты или сбора данных с него

3.В случае коректной работы всех систем, Для пользователей которые ушли с ошибкой, провести ретаргетинг с предложением о скидке + Email-рассылку


In [21]:
query = """
SELECT payment_id AS sc_fl
,(SELECT payment_id FROM payments WHERE payment_status = 'refunded' and payment_status = 'failed') AS rf_fl 
 FROM payments WHERE payment_status = 'success' and payment_status = 'failed' 

"""
pd.read_sql(query, conn)

,sc_fl,rf_fl


In [22]:
query = """
SELECT user_id 
,COUNT(payment_id) AS cnt_status
 FROM payments GROUP BY user_id   HAVING COUNT(payment_id) > 1

"""
pd.read_sql(query, conn)

,user_id,cnt_status


In [23]:
query = """
SELECT COUNT(*)

 FROM payments WHERE payment_status = 'refunded'

"""
pd.read_sql(query, conn)

,COUNT(*)
0,0


In [24]:
query = """
SELECT Round((1.0 *COUNT(*) / (SELECT COUNT(*) FROM payments) ) * 100 ,1) AS failed_percent
 FROM payments WHERE payment_status = 'failed'

"""
pd.read_sql(query, conn)

,failed_percent
0,22.7
